# Production-oriented text-first land geolocation pipeline (England & Wales)

This notebook implements a **fully automated, text-first** pipeline for complex land/property descriptions such as:

> Land adjoining 12 and 14 High Street, lying to the rear of 10 High Street and bounded on the east by Mill Lane.

It deliberately **does not use location plans in V1**. The input schema can keep a plan URI for a future fallback module, but the critical path is:

**description -> Azure OpenAI structured parsing -> OS anchor retrieval -> global ambiguity resolution -> HMLR INSPIRE candidate retrieval -> deterministic GIS evidence -> Azure OpenAI semantic adjudication -> conservative final decision**.

### Design goals

- Zero human intervention.
- Never ask the LLM to invent coordinates, UPRNs or HMLR polygons.
- Preserve multiple address hypotheses until geographic evidence resolves them.
- Treat missing evidence as **unknown**, not automatically false.
- Prefer `UNRESOLVED`/`MULTIPLE_CANDIDATES` over a confident wrong answer.
- Work inside AWS SageMaker with S3-backed Parquet/GeoParquet; no RDS/PostGIS server is required.
- Keep every important threshold in configuration so it can be calibrated from labelled examples.

### Important limitation

HMLR INSPIRE polygons are **indicative registered freehold extents**, not definitive legal title boundaries. Therefore the output is a ranked geolocation/candidate result, not a legal boundary determination.

In [ ]:
# Run once in a SageMaker kernel/environment.
%pip install -q -U \
    "openai>=2.0.0" "pydantic>=2.7" \
    boto3 s3fs fsspec pyarrow pandas numpy rapidfuzz \
    geopandas shapely pyproj

In [ ]:
import os
import re
import io
import json
import math
import hashlib
import uuid
import logging
from dataclasses import dataclass, field
from collections import Counter, defaultdict
from pathlib import Path
from typing import Any, Optional, Literal, Iterable

import boto3
import fsspec
import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq
import geopandas as gpd

from rapidfuzz import fuzz
from shapely.geometry import Point, LineString, MultiPoint, Polygon, box, shape, mapping
from shapely.ops import nearest_points, unary_union
from pyproj import Transformer

from pydantic import BaseModel, ConfigDict, Field
from openai import OpenAI

logging.basicConfig(level=logging.INFO, format="%(asctime)s | %(levelname)s | %(message)s")
logger = logging.getLogger("land-geolocator")

## 1. Configuration

The defaults are **starting values, not universal truths**. Tune them on labelled historical cases before treating the confidence score as calibrated.

In [ ]:
@dataclass
class S3Paths:
    # AddressBase Core canonical records + inverted token index.
    address_records: str = "s3://YOUR-BUCKET/processed/os/address_records/"
    address_tokens: str = "s3://YOUR-BUCKET/processed/os/address_tokens/"
    address_token_stats: Optional[str] = "s3://YOUR-BUCKET/processed/os/address_token_stats.parquet"

    # OS Open Names canonical records + token index.
    name_records: str = "s3://YOUR-BUCKET/processed/os/name_records/"
    name_tokens: str = "s3://YOUR-BUCKET/processed/os/name_tokens/"
    name_token_stats: Optional[str] = "s3://YOUR-BUCKET/processed/os/name_token_stats.parquet"

    # Prefer OS Open USRN for named-street geometry if available.
    # Fallback: OS Open Roads spatial links.
    street_geometries: Optional[str] = "s3://YOUR-BUCKET/processed/os/open_usrn/"
    road_geometries: Optional[str] = "s3://YOUR-BUCKET/processed/os/open_roads/"
    river_geometries: Optional[str] = "s3://YOUR-BUCKET/processed/os/open_rivers/"

    # HMLR INSPIRE GeoParquet, partitioned by 10 km tile.
    inspire: str = "s3://YOUR-BUCKET/processed/hmlr/inspire/"
    results: str = "s3://YOUR-BUCKET/results/"


@dataclass
class LLMConfig:
    endpoint: str = os.getenv("AZURE_OPENAI_ENDPOINT", "https://YOUR-RESOURCE.openai.azure.com")
    api_key: str = os.getenv("AZURE_OPENAI_API_KEY", "")
    deployment: str = os.getenv("AZURE_OPENAI_DEPLOYMENT", "YOUR_DEPLOYMENT")

    # Accuracy-oriented stages. Disable individually when optimising cost/latency.
    verify_parse: bool = True
    generate_fallback_search_variants: bool = True
    semantic_rerank: bool = True

    max_search_variants: int = 6
    max_semantic_candidates: int = 10


@dataclass
class RetrievalConfig:
    # S3 inverted-index layout. 3 hex chars = 4096 buckets.
    token_bucket_chars: int = 3
    record_bucket_chars: int = 2

    # Per-anchor retrieval limits.
    token_rows_cap: int = 25_000
    max_candidate_uprns_before_record_fetch: int = 800
    max_address_candidates_per_anchor: int = 25
    max_name_candidates_per_anchor: int = 20

    # Fuzzy matching thresholds are 0..1 after normalisation.
    min_anchor_lexical_score: float = 0.48
    strong_anchor_lexical_score: float = 0.78

    # Global ambiguity beam search.
    hypothesis_beam_width: int = 60
    max_anchor_hypotheses: int = 8
    unresolved_anchor_penalty: float = 0.22
    geographic_coherence_scale_m: float = 2500.0
    geographic_hard_cutoff_m: float = 30_000.0

    # Candidate polygon search.
    default_subject_search_radius_m: float = 180.0
    rural_subject_search_radius_m: float = 600.0
    max_subject_search_radius_m: float = 5000.0
    max_inspire_candidates_per_hypothesis: int = 250

    # Try to attach an INSPIRE extent to property/address anchors.
    anchor_parcel_search_radius_m: float = 60.0
    anchor_parcel_max_distance_m: float = 35.0


@dataclass
class SpatialConfig:
    # Relation tolerances. These are deliberately soft scoring scales.
    adjacent_boundary_scale_m: float = 8.0
    adjacent_point_scale_m: float = 30.0
    same_extent_adjacency_score: float = 0.20
    near_scale_m: float = 100.0
    between_corridor_m: float = 35.0
    direction_tolerance_deg: float = 55.0
    bounded_by_scale_m: float = 15.0
    rear_front_distance_scale_m: float = 120.0
    opposite_distance_scale_m: float = 120.0
    nearest_road_search_m: float = 150.0

    relation_weights: dict = field(default_factory=lambda: {
        "bounded_by": 2.0,
        "adjacent_to": 1.8,
        "between": 1.6,
        "rear_of": 1.4,
        "front_of": 1.2,
        "opposite": 1.2,
        "north_of": 0.9,
        "south_of": 0.9,
        "east_of": 0.9,
        "west_of": 0.9,
        "near": 0.6,
        "accessed_from": 0.7,
        "part_of": 0.8,
    })

    # Required constraint with a very low score is treated as a contradiction.
    required_contradiction_threshold: float = 0.12
    required_contradiction_multiplier: float = 0.45


@dataclass
class DecisionConfig:
    # Candidate score components. They should sum to 1.
    spatial_weight: float = 0.52
    anchor_hypothesis_weight: float = 0.20
    semantic_weight: float = 0.18
    evidence_coverage_weight: float = 0.10

    # Final conservative decision thresholds.
    matched_threshold: float = 0.78
    minimum_candidate_threshold: float = 0.45
    minimum_winner_margin: float = 0.08

    # Confidence also rewards winner separation and evidence completeness.
    confidence_score_weight: float = 0.65
    confidence_margin_weight: float = 0.20
    confidence_coverage_weight: float = 0.15


@dataclass
class AppConfig:
    aws_region: str = os.getenv("AWS_REGION", "eu-west-2")
    paths: S3Paths = field(default_factory=S3Paths)
    llm: LLMConfig = field(default_factory=LLMConfig)
    retrieval: RetrievalConfig = field(default_factory=RetrievalConfig)
    spatial: SpatialConfig = field(default_factory=SpatialConfig)
    decision: DecisionConfig = field(default_factory=DecisionConfig)


CONFIG = AppConfig()
CONFIG

In [ ]:
# Azure client. AZURE_OPENAI_DEPLOYMENT must be your deployment name.
azure_client = OpenAI(
    base_url=CONFIG.llm.endpoint.rstrip("/") + "/openai/v1/",
    api_key=CONFIG.llm.api_key,
)

s3_client = boto3.client("s3", region_name=CONFIG.aws_region)
BNG_TO_WGS84 = Transformer.from_crs("EPSG:27700", "EPSG:4326", always_xy=True)

## 2. Structured models

The parser explicitly supports any number of anchors. It also stores address components so the retrieval layer does not depend on the wording of one single free-text string.

In [ ]:
AnchorType = Literal[
    "address", "property", "road", "landmark", "watercourse",
    "railway", "settlement", "farm", "other", "unknown"
]
RelationType = Literal[
    "adjacent_to", "between", "rear_of", "front_of", "opposite",
    "north_of", "south_of", "east_of", "west_of", "bounded_by",
    "near", "accessed_from", "part_of", "other"
]
StrengthType = Literal["required", "strong", "medium", "weak"]


class AnchorSpec(BaseModel):
    model_config = ConfigDict(extra="forbid")
    id: str
    type: AnchorType
    raw_text: str

    house_number: Optional[str] = None
    building_name: Optional[str] = None
    street: Optional[str] = None
    locality: Optional[str] = None
    town: Optional[str] = None
    postcode: Optional[str] = None

    # Search alternatives are linguistic only (abbreviations/spelling), never coordinates.
    aliases: list[str] = Field(default_factory=list)
    notes: Optional[str] = None


class SpatialConstraint(BaseModel):
    model_config = ConfigDict(extra="forbid")
    relation: RelationType
    anchor_ids: list[str]
    direction: Optional[Literal["north", "south", "east", "west"]] = None
    strength: StrengthType = "medium"
    original_phrase: str


class ParsedDescription(BaseModel):
    model_config = ConfigDict(extra="forbid")
    subject_type: Literal[
        "land", "property", "building", "garage", "plot", "unit", "other", "unknown"
    ]
    subject_name: Optional[str] = None
    anchors: list[AnchorSpec]
    constraints: list[SpatialConstraint]
    locality_context: Optional[str] = None
    town_context: Optional[str] = None
    postcode_context: Optional[str] = None
    normalized_description: str
    ambiguity_notes: list[str]


class SearchVariantSet(BaseModel):
    model_config = ConfigDict(extra="forbid")
    variants: list[str]


class SemanticCandidate(BaseModel):
    model_config = ConfigDict(extra="forbid")
    inspire_id: str
    consistency_score: float = Field(ge=0, le=1)
    contradictions: list[str]
    supporting_points: list[str]


class SemanticAssessment(BaseModel):
    model_config = ConfigDict(extra="forbid")
    candidates: list[SemanticCandidate]
    notes: list[str]


class PipelineResult(BaseModel):
    model_config = ConfigDict(extra="allow")
    record_id: str
    status: Literal["MATCHED", "MULTIPLE_CANDIDATES", "UNRESOLVED", "INVALID_INPUT", "ERROR"]
    confidence_score: float = 0.0
    best_candidate: Optional[dict] = None
    candidates: list[dict] = Field(default_factory=list)
    parsed_description: Optional[dict] = None
    anchor_hypotheses: list[dict] = Field(default_factory=list)
    unresolved_reasons: list[str] = Field(default_factory=list)
    warnings: list[str] = Field(default_factory=list)
    errors: list[str] = Field(default_factory=list)

## 3. Text normalisation and indexing helpers

The runtime address search uses an **inverted token index in S3**. This avoids repeatedly scanning every AddressBase record in England and Wales.

In [ ]:
STREET_ABBREVIATIONS = {
    "rd": "road", "st": "street", "ln": "lane", "ave": "avenue",
    "av": "avenue", "dr": "drive", "ct": "court", "cl": "close",
    "cres": "crescent", "hwy": "highway", "pl": "place", "sq": "square",
    "terr": "terrace", "ter": "terrace"
}
STOP_TOKENS = {"the", "of", "at", "and", "land", "plot", "property"}


def normalize_text(value: Optional[str]) -> str:
    if value is None:
        return ""
    s = str(value).lower().strip()
    s = s.replace("&", " and ")
    s = re.sub(r"[^a-z0-9]+", " ", s)
    parts = [STREET_ABBREVIATIONS.get(p, p) for p in s.split()]
    return " ".join(parts)


def tokenise(value: Optional[str]) -> list[str]:
    return [t for t in normalize_text(value).split() if t and t not in STOP_TOKENS]


def stable_bucket(value: str, chars: int) -> str:
    return hashlib.sha1(value.encode("utf-8")).hexdigest()[:chars]


def uprn_bucket(uprn: str) -> str:
    return stable_bucket(str(uprn), CONFIG.retrieval.record_bucket_chars)


def token_bucket(token: str) -> str:
    return stable_bucket(token, CONFIG.retrieval.token_bucket_chars)


def tile10km(easting: float, northing: float) -> str:
    return f"E{int(float(easting)//10000):03d}_N{int(float(northing)//10000):03d}"


def tiles_for_bbox(bounds: tuple[float, float, float, float]) -> list[str]:
    minx, miny, maxx, maxy = bounds
    xs = range(int(minx // 10000), int(maxx // 10000) + 1)
    ys = range(int(miny // 10000), int(maxy // 10000) + 1)
    return [f"E{x:03d}_N{y:03d}" for x in xs for y in ys]


def is_postcode_like(value: str) -> bool:
    compact = normalize_text(value).replace(" ", "").upper()
    return bool(re.match(r"^[A-Z]{1,2}[0-9][A-Z0-9]?[0-9][A-Z]{2}$", compact))


def strength_multiplier(strength: str) -> float:
    return {"required": 1.35, "strong": 1.15, "medium": 1.0, "weak": 0.65}.get(strength, 1.0)


def exponential_score(distance: float, scale: float) -> float:
    return float(math.exp(-max(0.0, distance) / max(1e-6, scale)))


def bearing_deg(dx: float, dy: float) -> float:
    # 0=N, 90=E, 180=S, 270=W
    return (math.degrees(math.atan2(dx, dy)) + 360.0) % 360.0


def angular_difference(a: float, b: float) -> float:
    return abs((a - b + 180.0) % 360.0 - 180.0)


def json_safe(obj: Any) -> str:
    def default(x):
        if isinstance(x, np.generic):
            return x.item()
        if hasattr(x, "__geo_interface__"):
            return mapping(x)
        return str(x)
    return json.dumps(obj, default=default)

## 4. One-time preprocessing helpers

For national-scale use, run preprocessing as a dedicated SageMaker Processing job. The runtime expects:

- `address_records/uprn_bucket=../*.parquet`
- `address_tokens/token_bucket=.../*.parquet`
- `name_records/name_bucket=../*.parquet`
- `name_tokens/token_bucket=.../*.parquet`
- `street_geometries/tile10km=.../*.parquet` (prefer OS Open USRN)
- `road_geometries/tile10km=.../*.parquet` (fallback)
- `river_geometries/tile10km=.../*.parquet`
- `inspire/tile10km=.../*.parquet`

AddressBase Core CSV currently includes fields such as `UPRN`, `SINGLE_LINE_ADDRESS`, `STREET_NAME`, `TOWN_NAME`, `POST_TOWN`, `POSTCODE`, `X_COORDINATE` and `Y_COORDINATE`; the helper auto-detects common variants rather than assuming one delivery exactly.

In [ ]:
def detect_column(columns: Iterable[str], candidates: list[str], required: bool = True) -> Optional[str]:
    cols = list(columns)
    lower = {str(c).lower(): c for c in cols}
    for cand in candidates:
        if cand.lower() in lower:
            return lower[cand.lower()]
    for cand in candidates:
        for c in cols:
            if cand.lower() in str(c).lower():
                return c
    if required:
        raise KeyError(f"None of {candidates} found. Available columns include: {cols[:50]}")
    return None


def canonicalise_addressbase_chunk(df: pd.DataFrame) -> pd.DataFrame:
    c_uprn = detect_column(df.columns, ["UPRN", "uprn"])
    c_full = detect_column(df.columns, ["SINGLE_LINE_ADDRESS", "single_line_address", "full_address"])
    c_number = detect_column(df.columns, ["BUILDING_NUMBER", "building_number", "PAO_START_NUMBER"], required=False)
    c_bname = detect_column(df.columns, ["BUILDING_NAME", "building_name", "PAO_TEXT"], required=False)
    c_street = detect_column(df.columns, ["STREET_NAME", "street_name", "STREET_DESCRIPTION"], required=False)
    c_locality = detect_column(df.columns, ["LOCALITY", "locality", "DEPENDENT_LOCALITY"], required=False)
    c_town = detect_column(df.columns, ["POST_TOWN", "post_town", "TOWN_NAME", "town_name"], required=False)
    c_postcode = detect_column(df.columns, ["POSTCODE", "postcode", "POSTCODE_LOCATOR"], required=False)
    c_x = detect_column(df.columns, ["X_COORDINATE", "x_coordinate", "EASTING", "easting"])
    c_y = detect_column(df.columns, ["Y_COORDINATE", "y_coordinate", "NORTHING", "northing"])
    c_class = detect_column(df.columns, ["CLASSIFICATION_CODE", "classification_code"], required=False)

    def col(c):
        return df[c] if c else pd.Series([None] * len(df), index=df.index)

    out = pd.DataFrame({
        "uprn": col(c_uprn).astype("string"),
        "full_address": col(c_full).astype("string"),
        "house_number": col(c_number).astype("string"),
        "building_name": col(c_bname).astype("string"),
        "street": col(c_street).astype("string"),
        "locality": col(c_locality).astype("string"),
        "town": col(c_town).astype("string"),
        "postcode": col(c_postcode).astype("string"),
        "classification_code": col(c_class).astype("string"),
        "easting": pd.to_numeric(col(c_x), errors="coerce"),
        "northing": pd.to_numeric(col(c_y), errors="coerce"),
    })
    out = out.dropna(subset=["uprn", "full_address", "easting", "northing"]).copy()
    out["uprn_bucket"] = out["uprn"].map(uprn_bucket)
    return out


def address_record_tokens(row: pd.Series) -> list[tuple[str, str]]:
    values = []
    for kind, field in [
        ("house_number", "house_number"), ("building_name", "building_name"),
        ("street", "street"), ("locality", "locality"), ("town", "town"),
        ("postcode", "postcode")
    ]:
        for t in tokenise(row.get(field)):
            values.append((t, kind))
    # Full address catches components missing from explicit columns.
    for t in tokenise(row.get("full_address")):
        values.append((t, "full_address"))
    # de-duplicate token/kind pairs
    return list(dict.fromkeys(values))


def build_address_token_rows(records: pd.DataFrame) -> pd.DataFrame:
    rows=[]
    for _, r in records.iterrows():
        for token, kind in address_record_tokens(r):
            rows.append({
                "token": token,
                "token_bucket": token_bucket(token),
                "uprn": str(r["uprn"]),
                "kind": kind,
            })
    return pd.DataFrame(rows)


def write_partitioned_parquet(df: pd.DataFrame, base_uri: str, partition_col: str):
    # Simple generic writer suitable for batch preprocessing. Compact small files later.
    fs, root = fsspec.core.url_to_fs(base_uri)
    root = root.rstrip("/")
    for value, group in df.groupby(partition_col, dropna=False):
        rel = f"{root}/{partition_col}={value}/part-{uuid.uuid4().hex}.parquet"
        with fs.open(rel, "wb") as f:
            pq.write_table(pa.Table.from_pandas(group, preserve_index=False), f, compression="zstd")


def preprocess_addressbase_csv(input_csv: str, output_records: str, output_tokens: str, chunksize: int = 250_000):
    """One-time preprocessing. input_csv may be local or an fsspec-readable S3 CSV."""
    token_counts = Counter()
    for i, chunk in enumerate(pd.read_csv(input_csv, chunksize=chunksize, low_memory=False)):
        records = canonicalise_addressbase_chunk(chunk)
        tokens = build_address_token_rows(records)
        token_counts.update(tokens["token"].tolist())
        write_partitioned_parquet(records, output_records, "uprn_bucket")
        write_partitioned_parquet(tokens, output_tokens, "token_bucket")
        logger.info("AddressBase chunk %d: %d records / %d token rows", i, len(records), len(tokens))
    return pd.DataFrame({"token": list(token_counts), "document_frequency": list(token_counts.values())})

In [ ]:
def canonicalise_open_names(df: pd.DataFrame) -> pd.DataFrame:
    c_id = detect_column(df.columns, ["ID", "id", "NAMES_URI", "names_uri", "identifier"], required=False)
    c_name1 = detect_column(df.columns, ["NAME1", "name1", "name"])
    c_name2 = detect_column(df.columns, ["NAME2", "name2"], required=False)
    c_type = detect_column(df.columns, ["LOCAL_TYPE", "local_type", "TYPE", "type"], required=False)
    c_x = detect_column(df.columns, ["GEOMETRY_X", "geometry_x", "EASTING", "easting", "X", "x"])
    c_y = detect_column(df.columns, ["GEOMETRY_Y", "geometry_y", "NORTHING", "northing", "Y", "y"])

    def col(c):
        return df[c] if c else pd.Series([None] * len(df), index=df.index)

    out = pd.DataFrame({
        "name_id": col(c_id).astype("string") if c_id else [uuid.uuid4().hex for _ in range(len(df))],
        "name": col(c_name1).astype("string"),
        "alt_name": col(c_name2).astype("string"),
        "feature_type": col(c_type).astype("string"),
        "easting": pd.to_numeric(col(c_x), errors="coerce"),
        "northing": pd.to_numeric(col(c_y), errors="coerce"),
    }).dropna(subset=["name", "easting", "northing"])
    out["name_bucket"] = out["name_id"].astype(str).map(lambda x: stable_bucket(x, CONFIG.retrieval.record_bucket_chars))
    return out


def build_name_token_rows(records: pd.DataFrame) -> pd.DataFrame:
    rows=[]
    for _, r in records.iterrows():
        for field, kind in [("name", "name"), ("alt_name", "alt_name")]:
            for t in tokenise(r.get(field)):
                rows.append({
                    "token": t,
                    "token_bucket": token_bucket(t),
                    "name_id": str(r["name_id"]),
                    "kind": kind,
                })
    return pd.DataFrame(rows).drop_duplicates()


def preprocess_open_names_csv(input_csv: str, output_records: str, output_tokens: str):
    raw = pd.read_csv(input_csv, low_memory=False)
    records = canonicalise_open_names(raw)
    tokens = build_name_token_rows(records)
    write_partitioned_parquet(records, output_records, "name_bucket")
    write_partitioned_parquet(tokens, output_tokens, "token_bucket")
    return tokens.groupby("token").size().rename("document_frequency").reset_index()


def preprocess_vector_file(input_path: str, output_uri: str, id_candidates=None, name_candidates=None):
    """Generic OS street/road/river vector -> EPSG:27700 GeoParquet tiled at 10 km."""
    id_candidates = id_candidates or ["usrn", "USRN", "id", "ID", "identifier", "guid", "GUID"]
    name_candidates = name_candidates or ["street", "STREET", "name", "NAME", "road_name", "ROAD_NAME"]
    gdf = gpd.read_file(input_path)
    if gdf.crs is None:
        raise ValueError(f"Vector file has no CRS: {input_path}")
    gdf = gdf.to_crs(27700)
    id_col = detect_column(gdf.columns, id_candidates, required=False)
    name_col = detect_column(gdf.columns, name_candidates, required=False)
    gdf["feature_id"] = gdf[id_col].astype(str) if id_col else [uuid.uuid4().hex for _ in range(len(gdf))]
    gdf["name"] = gdf[name_col].astype(str) if name_col else ""
    cent = gdf.geometry.centroid
    gdf["tile10km"] = [tile10km(x, y) for x, y in zip(cent.x, cent.y)]
    for t, group in gdf[["feature_id", "name", "tile10km", "geometry"]].groupby("tile10km"):
        uri = output_uri.rstrip("/") + f"/tile10km={t}/part-{uuid.uuid4().hex}.parquet"
        group.to_parquet(uri, index=False)


def preprocess_hmlr_inspire_gml(input_gml: str, output_uri: str):
    gdf = gpd.read_file(input_gml)
    if gdf.crs is None:
        gdf = gdf.set_crs(27700)
    else:
        gdf = gdf.to_crs(27700)
    id_col = detect_column(
        gdf.columns,
        ["INSPIREID", "INSPIRE_ID", "inspire_id", "inspireId", "localId", "identifier"],
        required=False,
    )
    gdf["inspire_id"] = gdf[id_col].astype(str) if id_col else [uuid.uuid4().hex for _ in range(len(gdf))]
    bounds = gdf.geometry.bounds
    for c in ["minx", "miny", "maxx", "maxy"]:
        gdf[c] = bounds[c]
    cent = gdf.geometry.centroid
    gdf["tile10km"] = [tile10km(x, y) for x, y in zip(cent.x, cent.y)]
    keep = ["inspire_id", "minx", "miny", "maxx", "maxy", "tile10km", "geometry"]
    for t, group in gdf[keep].groupby("tile10km"):
        uri = output_uri.rstrip("/") + f"/tile10km={t}/part-{uuid.uuid4().hex}.parquet"
        group.to_parquet(uri, index=False)

## 5. S3/Parquet repositories

These classes contain the geographic truth. Azure OpenAI never substitutes its memory for these results.

In [ ]:
class PartitionStore:
    def __init__(self, base_uri: str):
        self.base_uri = base_uri.rstrip("/")
        self.fs, self.root = fsspec.core.url_to_fs(self.base_uri)
        self.root = self.root.rstrip("/")

    def glob_partition(self, partition_col: str, value: str) -> list[str]:
        pattern = f"{self.root}/{partition_col}={value}/*.parquet"
        return sorted(self.fs.glob(pattern))

    def read_partition_df(self, partition_col: str, value: str, columns: Optional[list[str]] = None) -> pd.DataFrame:
        frames=[]
        for p in self.glob_partition(partition_col, value):
            with self.fs.open(p, "rb") as f:
                frames.append(pd.read_parquet(f, columns=columns))
        return pd.concat(frames, ignore_index=True) if frames else pd.DataFrame(columns=columns or [])

    def read_partition_gdf(self, partition_col: str, value: str) -> gpd.GeoDataFrame:
        frames=[]
        for p in self.glob_partition(partition_col, value):
            # geopandas can read file-like parquet streams.
            with self.fs.open(p, "rb") as f:
                frames.append(gpd.read_parquet(f))
        if not frames:
            return gpd.GeoDataFrame(columns=["geometry"], geometry="geometry", crs=27700)
        g = gpd.GeoDataFrame(pd.concat(frames, ignore_index=True), geometry="geometry")
        if g.crs is None:
            g = g.set_crs(27700)
        elif g.crs.to_epsg() != 27700:
            g = g.to_crs(27700)
        return g


class TokenStats:
    def __init__(self, uri: Optional[str]):
        self.freq = {}
        if uri:
            try:
                df = pd.read_parquet(uri)
                self.freq = dict(zip(df["token"].astype(str), df["document_frequency"].astype(float)))
            except Exception as exc:
                logger.warning("Token stats unavailable at %s: %s", uri, exc)

    def rarity(self, token: str) -> float:
        # Unknown tokens are treated as relatively selective.
        f = self.freq.get(token, 10.0)
        return 1.0 / math.log10(max(10.0, f) + 10.0)

In [ ]:
class AddressRepository:
    def __init__(self, config: AppConfig):
        self.config = config
        self.records = PartitionStore(config.paths.address_records)
        self.tokens = PartitionStore(config.paths.address_tokens)
        self.stats = TokenStats(config.paths.address_token_stats)

    def _lookup_token(self, token: str) -> pd.DataFrame:
        b = token_bucket(token)
        df = self.tokens.read_partition_df("token_bucket", b, columns=["token", "uprn", "kind"])
        if df.empty:
            return df
        out = df[df["token"].astype(str) == token].copy()
        if len(out) > self.config.retrieval.token_rows_cap:
            out = out.head(self.config.retrieval.token_rows_cap)
        return out

    def _fetch_records(self, uprns: list[str]) -> pd.DataFrame:
        by_bucket=defaultdict(list)
        for u in uprns:
            by_bucket[uprn_bucket(str(u))].append(str(u))
        frames=[]
        wanted_cols = [
            "uprn", "full_address", "house_number", "building_name", "street",
            "locality", "town", "postcode", "classification_code", "easting", "northing"
        ]
        for b, ids in by_bucket.items():
            df = self.records.read_partition_df("uprn_bucket", b, columns=wanted_cols)
            if not df.empty:
                frames.append(df[df["uprn"].astype(str).isin(ids)])
        return pd.concat(frames, ignore_index=True) if frames else pd.DataFrame(columns=wanted_cols)

    def query_tokens(self, anchor: AnchorSpec, parsed: ParsedDescription, free_text_variant: Optional[str] = None) -> list[tuple[str, float]]:
        token_weights=[]
        fields = [
            (anchor.postcode or parsed.postcode_context, 4.0),
            (anchor.house_number, 3.2),
            (anchor.building_name, 2.7),
            (anchor.street, 2.5),
            (anchor.locality or parsed.locality_context, 1.5),
            (anchor.town or parsed.town_context, 1.7),
            (free_text_variant or anchor.raw_text, 1.0),
        ]
        for value, base_weight in fields:
            for t in tokenise(value):
                kind_bonus = 1.5 if (t.isdigit() or is_postcode_like(t)) else 1.0
                token_weights.append((t, base_weight * kind_bonus * self.stats.rarity(t)))
        # retain max weight per token
        best={}
        for t,w in token_weights:
            best[t]=max(best.get(t,0),w)
        return sorted(best.items(), key=lambda x: x[1], reverse=True)

    def search(self, anchor: AnchorSpec, parsed: ParsedDescription, variants: list[str] | None = None) -> list[dict]:
        variants = variants or [anchor.raw_text]
        hit_scores=defaultdict(float)
        hit_counts=Counter()

        for variant in variants:
            qtokens = self.query_tokens(anchor, parsed, variant)
            # Prefer the most selective few tokens. Reading every common word is wasteful.
            for token, weight in qtokens[:7]:
                rows = self._lookup_token(token)
                for u in rows.get("uprn", pd.Series(dtype=str)).astype(str):
                    hit_scores[u] += weight
                    hit_counts[u] += 1

        if not hit_scores:
            return []

        ranked_ids = [u for u,_ in sorted(hit_scores.items(), key=lambda kv: kv[1], reverse=True)]
        ranked_ids = ranked_ids[: self.config.retrieval.max_candidate_uprns_before_record_fetch]
        records = self._fetch_records(ranked_ids)
        if records.empty:
            return []

        def lexical_score(r) -> float:
            score = 0.0
            weight = 0.0

            # Full-text similarity to best variant.
            full_norm = normalize_text(r.get("full_address"))
            var_score = max(fuzz.WRatio(normalize_text(v), full_norm) for v in variants) / 100.0
            score += var_score * 0.40; weight += 0.40

            def component(query, actual, w, exact=False):
                nonlocal score, weight
                if not query:
                    return
                q, a = normalize_text(query), normalize_text(actual)
                if not a:
                    weight += w
                    return
                s = 1.0 if (exact and q == a) else fuzz.WRatio(q, a) / 100.0
                score += s*w; weight += w

            component(anchor.house_number, r.get("house_number"), 0.18, exact=True)
            component(anchor.building_name, r.get("building_name"), 0.12)
            component(anchor.street, r.get("street"), 0.14)
            component(anchor.town or parsed.town_context, r.get("town"), 0.08)
            component(anchor.postcode or parsed.postcode_context, r.get("postcode"), 0.08, exact=True)

            base = score / max(weight, 1e-9)
            # Small deterministic retrieval bonus based on token evidence.
            u = str(r["uprn"])
            token_bonus = min(0.08, 0.015 * hit_counts[u])
            return min(1.0, base + token_bonus)

        records["lexical_score"] = records.apply(lexical_score, axis=1)
        records = records.sort_values("lexical_score", ascending=False)
        records = records[records["lexical_score"] >= self.config.retrieval.min_anchor_lexical_score]
        records = records.head(self.config.retrieval.max_address_candidates_per_anchor)

        return [
            {
                "source": "OS_ADDRESSBASE_CORE",
                "source_id": str(r.uprn),
                "uprn": str(r.uprn),
                "label": str(r.full_address),
                "easting": float(r.easting),
                "northing": float(r.northing),
                "lexical_score": float(r.lexical_score),
                "postcode": None if pd.isna(r.postcode) else str(r.postcode),
                "town": None if pd.isna(r.town) else str(r.town),
                "street": None if pd.isna(r.street) else str(r.street),
            }
            for _, r in records.iterrows()
        ]

In [ ]:
class NameRepository:
    def __init__(self, config: AppConfig):
        self.config = config
        self.records = PartitionStore(config.paths.name_records)
        self.tokens = PartitionStore(config.paths.name_tokens)
        self.stats = TokenStats(config.paths.name_token_stats)

    def _lookup_token(self, token: str) -> pd.DataFrame:
        df = self.tokens.read_partition_df("token_bucket", token_bucket(token), columns=["token", "name_id", "kind"])
        if df.empty:
            return df
        return df[df["token"].astype(str) == token].head(self.config.retrieval.token_rows_cap)

    def _fetch_records(self, ids: list[str]) -> pd.DataFrame:
        by_bucket=defaultdict(list)
        for i in ids:
            by_bucket[stable_bucket(i, self.config.retrieval.record_bucket_chars)].append(i)
        frames=[]
        cols=["name_id", "name", "alt_name", "feature_type", "easting", "northing"]
        for b, subset in by_bucket.items():
            df=self.records.read_partition_df("name_bucket", b, columns=cols)
            if not df.empty:
                frames.append(df[df["name_id"].astype(str).isin(subset)])
        return pd.concat(frames, ignore_index=True) if frames else pd.DataFrame(columns=cols)

    def search(self, query: str, type_hint: Optional[str] = None) -> list[dict]:
        toks=tokenise(query)
        toks=sorted(set(toks), key=lambda t: self.stats.rarity(t), reverse=True)[:6]
        hits=defaultdict(float)
        for t in toks:
            weight=self.stats.rarity(t)
            rows=self._lookup_token(t)
            for i in rows.get("name_id", pd.Series(dtype=str)).astype(str):
                hits[i]+=weight
        if not hits:
            return []
        ids=[i for i,_ in sorted(hits.items(), key=lambda kv:kv[1], reverse=True)[:600]]
        df=self._fetch_records(ids)
        if df.empty:
            return []
        q=normalize_text(query)
        hint=normalize_text(type_hint)
        def score(r):
            s=max(
                fuzz.WRatio(q, normalize_text(r.get("name"))),
                fuzz.WRatio(q, normalize_text(r.get("alt_name"))) if pd.notna(r.get("alt_name")) else 0
            )/100.0
            if hint and hint in normalize_text(r.get("feature_type")):
                s=min(1.0, s+0.08)
            return s
        df["lexical_score"]=df.apply(score, axis=1)
        df=df[df.lexical_score>=self.config.retrieval.min_anchor_lexical_score].sort_values("lexical_score", ascending=False)
        return [
            {
                "source":"OS_OPEN_NAMES", "source_id":str(r.name_id), "label":str(r["name"]),
                "feature_type":str(r.feature_type), "easting":float(r.easting), "northing":float(r.northing),
                "lexical_score":float(r.lexical_score),
            }
            for _,r in df.head(self.config.retrieval.max_name_candidates_per_anchor).iterrows()
        ]


class VectorTileRepository:
    def __init__(self, uri: Optional[str]):
        self.store = PartitionStore(uri) if uri else None

    def available(self) -> bool:
        return self.store is not None

    def in_bbox(self, bounds: tuple) -> gpd.GeoDataFrame:
        if self.store is None:
            return gpd.GeoDataFrame(columns=["geometry"], geometry="geometry", crs=27700)
        frames=[]
        for t in tiles_for_bbox(bounds):
            g=self.store.read_partition_gdf("tile10km", t)
            if not g.empty:
                frames.append(g)
        if not frames:
            return gpd.GeoDataFrame(columns=["geometry"], geometry="geometry", crs=27700)
        out=gpd.GeoDataFrame(pd.concat(frames, ignore_index=True), geometry="geometry", crs=27700)
        return out[out.intersects(box(*bounds))].copy()


class InspireRepository(VectorTileRepository):
    def __init__(self, config: AppConfig):
        super().__init__(config.paths.inspire)
        self.config=config

    def candidates(self, bounds: tuple, limit: Optional[int]=None) -> gpd.GeoDataFrame:
        g=self.in_bbox(bounds)
        if g.empty:
            return g
        qbox=box(*bounds)
        g=g[g.intersects(qbox)].copy()
        if limit and len(g)>limit:
            c=qbox.centroid
            g["_d"]=g.geometry.distance(c)
            g=g.sort_values("_d").head(limit).drop(columns="_d")
        return g

## 6. Azure OpenAI parser, verifier and fallback search variants

The LLM is used aggressively for **language understanding**, but not for geographic truth.

In [ ]:
PARSE_PROMPT = """
You parse complex land and property location descriptions for England and Wales.

Return a complete structured representation of the SUBJECT, all geographic ANCHORS,
and all spatial CONSTRAINTS from the subject to those anchors.

Rules:
- Never invent coordinates, UPRNs, title numbers, postcodes, towns or roads.
- A referenced address/property is an anchor, not automatically the subject address.
- Expand shared grammar carefully. Example: 'between 10 and 14 Station Road, Chester'
  means two address anchors: 10 Station Road, Chester and 14 Station Road, Chester.
- 'rear of numbers 10 to 16 High Street' may require anchors for the explicitly described
  range if the wording clearly enumerates/defines them. Do not invent odd/even numbers not stated.
- Preserve named farms, cottages, churches, roads, rivers, railways and settlements.
- Use aliases only for spelling/abbreviation variants of text that is already present.
- Mark uncertainty in ambiguity_notes instead of guessing.
- Normalize relations: adjoining/adjoins=adjacent_to; behind/at rear of=rear_of;
  opposite=opposite; bounded by=bounded_by.
- bounded_by.direction means the side of the SUBJECT boundary, e.g. 'bounded on the east by Mill Lane'
  -> direction='east'.
"""

VERIFY_PROMPT = """
You are a strict verifier of an already parsed England/Wales land description.
Compare the original description and proposed structured parse. Return a corrected COMPLETE parse.
Preserve the proposed parse when it is supported. Fix only omissions, incorrect shared-address
expansion, wrong relation direction, or facts not actually stated. Never add geographic facts from memory.
"""

SEARCH_VARIANT_PROMPT = """
Generate linguistic search variants for ONE unresolved geographic anchor from a land description.
Variants may normalize abbreviations, punctuation, word order, house-number formatting or obvious OCR-like spacing.
Do NOT add a town, postcode, road, landmark or property name that was not present in the supplied anchor/context.
Do NOT generate coordinates. Return at most the requested number of variants.
"""


class LLMServices:
    def __init__(self, client: OpenAI, config: AppConfig):
        self.client=client; self.config=config

    def parse_description(self, description: str) -> ParsedDescription:
        r=self.client.responses.parse(
            model=self.config.llm.deployment,
            instructions=PARSE_PROMPT,
            input=description,
            text_format=ParsedDescription,
        )
        parsed=r.output_parsed
        if not self.config.llm.verify_parse:
            return parsed
        vr=self.client.responses.parse(
            model=self.config.llm.deployment,
            instructions=VERIFY_PROMPT,
            input=json_safe({"original":description, "proposed":parsed.model_dump()}),
            text_format=ParsedDescription,
        )
        return vr.output_parsed

    def search_variants(self, anchor: AnchorSpec, parsed: ParsedDescription) -> list[str]:
        if not self.config.llm.generate_fallback_search_variants:
            return []
        payload={
            "anchor":anchor.model_dump(),
            "shared_context":{
                "locality":parsed.locality_context,
                "town":parsed.town_context,
                "postcode":parsed.postcode_context,
            },
            "maximum_variants":self.config.llm.max_search_variants,
        }
        r=self.client.responses.parse(
            model=self.config.llm.deployment,
            instructions=SEARCH_VARIANT_PROMPT,
            input=json_safe(payload),
            text_format=SearchVariantSet,
        )
        seen=set(); out=[]
        for v in r.output_parsed.variants:
            n=normalize_text(v)
            if n and n not in seen:
                seen.add(n); out.append(v)
        return out[:self.config.llm.max_search_variants]

## 7. Anchor retrieval and global ambiguity resolution

A critical accuracy rule: **do not resolve each ambiguous address independently**. The beam search below maintains several whole-description geographic hypotheses and rewards candidates that form a coherent cluster.

In [ ]:
@dataclass
class AnchorCandidate:
    anchor_id: str
    anchor_type: str
    source: str
    source_id: str
    label: str
    easting: float
    northing: float
    lexical_score: float
    uprn: Optional[str]=None
    feature_type: Optional[str]=None
    geometry_geojson: Optional[dict]=None
    parcel_geojson: Optional[dict]=None

    @property
    def point(self):
        return Point(self.easting, self.northing)


@dataclass
class AnchorHypothesis:
    selections: dict[str, Optional[AnchorCandidate]]
    raw_score: float
    normalized_score: float=0.0


def candidate_from_dict(anchor: AnchorSpec, d: dict) -> AnchorCandidate:
    return AnchorCandidate(
        anchor_id=anchor.id, anchor_type=anchor.type,
        source=d["source"], source_id=str(d["source_id"]), label=d["label"],
        easting=float(d["easting"]), northing=float(d["northing"]), lexical_score=float(d["lexical_score"]),
        uprn=d.get("uprn"), feature_type=d.get("feature_type"), geometry_geojson=d.get("geometry_geojson")
    )


class AnchorResolver:
    def __init__(self, addresses: AddressRepository, names: NameRepository, llm: LLMServices, config: AppConfig):
        self.addresses=addresses; self.names=names; self.llm=llm; self.config=config

    def retrieve_anchor(self, anchor: AnchorSpec, parsed: ParsedDescription) -> list[AnchorCandidate]:
        if anchor.type in {"address", "property", "farm"}:
            rows=self.addresses.search(anchor, parsed, variants=[anchor.raw_text] + list(anchor.aliases))
            if (not rows or rows[0]["lexical_score"] < self.config.retrieval.strong_anchor_lexical_score):
                variants=self.llm.search_variants(anchor, parsed)
                if variants:
                    rows=self.addresses.search(anchor, parsed, variants=[anchor.raw_text] + variants)
        else:
            query=" ".join(x for x in [anchor.raw_text, anchor.town or parsed.town_context] if x)
            rows=self.names.search(query, type_hint=anchor.type)
            if (not rows or rows[0]["lexical_score"] < self.config.retrieval.strong_anchor_lexical_score):
                for variant in self.llm.search_variants(anchor, parsed):
                    extra=self.names.search(variant, type_hint=anchor.type)
                    rows=(rows+extra)
                # de-duplicate
                best={}
                for r in rows:
                    k=(r["source"],r["source_id"])
                    if k not in best or r["lexical_score"]>best[k]["lexical_score"]:
                        best[k]=r
                rows=sorted(best.values(), key=lambda r:r["lexical_score"], reverse=True)
        return [candidate_from_dict(anchor,r) for r in rows]

    def retrieve_all(self, parsed: ParsedDescription) -> dict[str,list[AnchorCandidate]]:
        return {a.id:self.retrieve_anchor(a,parsed) for a in parsed.anchors}

    def _pair_scale(self, a: AnchorCandidate, b: AnchorCandidate) -> float:
        # Settlement centroids can legitimately be farther from the subject than property anchors.
        if "settlement" in {a.anchor_type,b.anchor_type}:
            return self.config.retrieval.geographic_coherence_scale_m*3.0
        if {a.anchor_type,b.anchor_type} & {"road","watercourse","railway"}:
            return self.config.retrieval.geographic_coherence_scale_m*1.5
        return self.config.retrieval.geographic_coherence_scale_m

    def _coherence_increment(self, candidate: AnchorCandidate, selected: dict[str,Optional[AnchorCandidate]]) -> float:
        others=[x for x in selected.values() if x is not None]
        if not others:
            return 0.0
        vals=[]
        for o in others:
            d=candidate.point.distance(o.point)
            if d>self.config.retrieval.geographic_hard_cutoff_m and "settlement" not in {candidate.anchor_type,o.anchor_type}:
                vals.append(-1.0)
            else:
                vals.append(exponential_score(d, self._pair_scale(candidate,o)))
        return float(np.mean(vals)) if vals else 0.0

    def build_hypotheses(self, candidate_sets: dict[str,list[AnchorCandidate]]) -> list[AnchorHypothesis]:
        # Most selective anchors first limits combinatorial explosion.
        order=sorted(candidate_sets, key=lambda aid: (len(candidate_sets[aid]) if candidate_sets[aid] else 10_000))
        beam=[AnchorHypothesis(selections={}, raw_score=0.0)]

        for aid in order:
            options=candidate_sets[aid][:self.config.retrieval.max_address_candidates_per_anchor]
            # An unresolved option lets the pipeline continue rather than crash/force a false anchor.
            extended=[]
            for h in beam:
                if not options:
                    s=dict(h.selections); s[aid]=None
                    extended.append(AnchorHypothesis(s, h.raw_score-self.config.retrieval.unresolved_anchor_penalty))
                else:
                    for c in options:
                        coherence=self._coherence_increment(c,h.selections)
                        # Lexical score is primary; coherence is a strong secondary signal.
                        inc=0.72*c.lexical_score + 0.28*coherence
                        s=dict(h.selections); s[aid]=c
                        extended.append(AnchorHypothesis(s,h.raw_score+inc))
                    # Keep an unresolved branch when all candidates may be misleading.
                    s=dict(h.selections); s[aid]=None
                    extended.append(AnchorHypothesis(s,h.raw_score-self.config.retrieval.unresolved_anchor_penalty))
            beam=sorted(extended,key=lambda h:h.raw_score,reverse=True)[:self.config.retrieval.hypothesis_beam_width]

        beam=beam[:self.config.retrieval.max_anchor_hypotheses]
        if not beam:
            return []
        raw=np.array([h.raw_score for h in beam],dtype=float)
        lo,hi=float(raw.min()),float(raw.max())
        for h in beam:
            h.normalized_score=1.0 if hi==lo else (h.raw_score-lo)/(hi-lo)
        return beam

## 8. Enrich anchors with local street/river geometry and nearby INSPIRE extent

For a property anchor, the UPRN is a point. For relations such as **adjacent to**, it is much better when possible to identify the nearby INSPIRE polygon containing that point and use polygon-to-polygon geometry rather than point distance alone.

In [ ]:
class GeographicContext:
    def __init__(self, config: AppConfig, inspire: InspireRepository):
        self.config=config; self.inspire=inspire
        self.streets=VectorTileRepository(config.paths.street_geometries)
        self.roads=VectorTileRepository(config.paths.road_geometries)
        self.rivers=VectorTileRepository(config.paths.river_geometries)

    def _nearest_line(self, point: Point, repo: VectorTileRepository, radius: float, name: Optional[str]=None):
        bounds=(point.x-radius, point.y-radius, point.x+radius, point.y+radius)
        g=repo.in_bbox(bounds)
        if g.empty:
            return None
        if name and "name" in g.columns and g["name"].fillna("").str.len().gt(0).any():
            g=g.copy()
            g["_name_score"]=g["name"].fillna("").map(lambda v:fuzz.WRatio(normalize_text(name), normalize_text(v)))
            # Keep plausible name matches; if none, fall back to nearest geometry.
            plausible=g[g["_name_score"]>=65]
            if not plausible.empty:
                g=plausible
        d=g.geometry.distance(point)
        return g.loc[d.idxmin()].geometry

    def enrich_candidate(self, c: AnchorCandidate, anchor: AnchorSpec) -> AnchorCandidate:
        p=c.point
        # Attach named linear geometry when possible.
        if anchor.type=="road":
            geom=None
            if self.streets.available():
                geom=self._nearest_line(p,self.streets,300,anchor.raw_text)
            if geom is None and self.roads.available():
                geom=self._nearest_line(p,self.roads,200,None)
            if geom is not None:
                c.geometry_geojson=mapping(geom)
        elif anchor.type=="watercourse" and self.rivers.available():
            geom=self._nearest_line(p,self.rivers,1200,anchor.raw_text)
            if geom is not None:
                c.geometry_geojson=mapping(geom)

        # Attach indicative anchor parcel for property/address/farm anchors.
        if anchor.type in {"address","property","farm"}:
            r=self.config.retrieval.anchor_parcel_search_radius_m
            g=self.inspire.candidates((p.x-r,p.y-r,p.x+r,p.y+r),limit=30)
            if not g.empty:
                contains=g[g.geometry.covers(p)]
                if not contains.empty:
                    chosen=contains.iloc[0].geometry
                    c.parcel_geojson=mapping(chosen)
                else:
                    d=g.geometry.distance(p)
                    idx=d.idxmin()
                    if float(d.loc[idx])<=self.config.retrieval.anchor_parcel_max_distance_m:
                        c.parcel_geojson=mapping(g.loc[idx].geometry)
        return c

    def enrich_hypothesis(self, h: AnchorHypothesis, parsed: ParsedDescription) -> AnchorHypothesis:
        amap={a.id:a for a in parsed.anchors}
        for aid,c in h.selections.items():
            if c is not None:
                self.enrich_candidate(c,amap[aid])
        return h

    def nearest_road_to_property(self, c: AnchorCandidate):
        p=c.point
        if self.streets.available():
            g=self._nearest_line(p,self.streets,self.config.spatial.nearest_road_search_m,None)
            if g is not None:
                return g
        if self.roads.available():
            return self._nearest_line(p,self.roads,self.config.spatial.nearest_road_search_m,None)
        return None

## 9. Adaptive INSPIRE search area

In [ ]:
def hypothesis_search_bbox(h: AnchorHypothesis, parsed: ParsedDescription, config: AppConfig) -> Optional[tuple]:
    selected=[c for c in h.selections.values() if c is not None]
    if not selected:
        return None
    pts=[c.point for c in selected]
    if len(pts)==1:
        radius=config.retrieval.default_subject_search_radius_m
        if selected[0].anchor_type in {"settlement","farm","landmark"}:
            radius=config.retrieval.rural_subject_search_radius_m
    else:
        mp=MultiPoint(pts)
        minx,miny,maxx,maxy=mp.bounds
        spread=math.hypot(maxx-minx,maxy-miny)
        radius=max(config.retrieval.default_subject_search_radius_m, spread*0.65)
        if any(c.anchor_type in {"settlement","farm"} for c in selected):
            radius=max(radius,config.retrieval.rural_subject_search_radius_m)
    radius=min(radius,config.retrieval.max_subject_search_radius_m)
    xs=[p.x for p in pts]; ys=[p.y for p in pts]
    return (min(xs)-radius,min(ys)-radius,max(xs)+radius,max(ys)+radius)

## 10. Deterministic spatial relation engine

Each relation returns a score **and an `available` flag**. If the data needed to test a relation is absent, that clue is not treated as a contradiction.

In [ ]:
class SpatialEngine:
    def __init__(self, context: GeographicContext, config: AppConfig):
        self.ctx=context; self.config=config

    @staticmethod
    def anchor_geometry(c: AnchorCandidate, prefer_parcel=True):
        if prefer_parcel and c.parcel_geojson:
            return shape(c.parcel_geojson)
        if c.geometry_geojson:
            return shape(c.geometry_geojson)
        return c.point

    def adjacent(self, candidate, anchors):
        if not anchors: return 0.0,False,{"reason":"no resolved anchors"}
        vals=[]; details=[]
        for a in anchors:
            g=self.anchor_geometry(a,prefer_parcel=True)
            d=candidate.distance(g)
            same_extent = (not isinstance(g, Point)) and candidate.equals(g)
            if same_extent:
                score = self.config.spatial.same_extent_adjacency_score
            else:
                scale=self.config.spatial.adjacent_boundary_scale_m if not isinstance(g,Point) else self.config.spatial.adjacent_point_scale_m
                score=exponential_score(d,scale)
            vals.append(score); details.append({"distance_m":float(d),"same_indicative_extent":bool(same_extent)})
        return float(np.mean(vals)),True,{"anchors":details}

    def near(self,candidate,anchors):
        if not anchors:return 0.0,False,{"reason":"no resolved anchors"}
        ds=[candidate.distance(self.anchor_geometry(a)) for a in anchors]
        return float(np.mean([exponential_score(d,self.config.spatial.near_scale_m) for d in ds])),True,{"distances_m":[float(d) for d in ds]}

    def between(self,candidate,anchors):
        if len(anchors)<2:return 0.0,False,{"reason":"between requires >=2 resolved anchors"}
        pts=[self.anchor_geometry(a).centroid for a in anchors]
        if len(pts)==2:
            zone=LineString(pts).buffer(self.config.spatial.between_corridor_m)
        else:
            zone=MultiPoint(pts).convex_hull.buffer(self.config.spatial.between_corridor_m)
        overlap=candidate.intersection(zone).area
        ratio=overlap/candidate.area if candidate.area>0 else 0.0
        centroid_inside=zone.covers(candidate.centroid)
        score=min(1.0,0.72*ratio+0.28*float(centroid_inside))
        return float(score),True,{"zone_overlap_ratio":float(ratio),"centroid_in_between_zone":bool(centroid_inside)}

    def directional(self,candidate,anchors,desired):
        if not anchors:return 0.0,False,{"reason":"no resolved anchors"}
        target={"north":0,"east":90,"south":180,"west":270}[desired]
        vals=[]; bearings=[]
        for a in anchors:
            p=self.anchor_geometry(a).centroid; c=candidate.centroid
            b=bearing_deg(c.x-p.x,c.y-p.y); bearings.append(float(b))
            diff=angular_difference(b,target)
            vals.append(max(0.0,1.0-diff/self.config.spatial.direction_tolerance_deg))
        return float(np.mean(vals)),True,{"bearings_deg":bearings}

    def bounded_by(self,candidate,anchors,direction=None):
        if not anchors:return 0.0,False,{"reason":"no resolved anchors"}
        desired={"north":0,"east":90,"south":180,"west":270}.get(direction)
        vals=[]; details=[]
        for a in anchors:
            g=self.anchor_geometry(a,prefer_parcel=False)
            d=candidate.boundary.distance(g)
            dscore=exponential_score(d,self.config.spatial.bounded_by_scale_m)
            dirscore=1.0; b=None
            if desired is not None:
                cc=candidate.centroid; ac=g.centroid
                b=bearing_deg(ac.x-cc.x,ac.y-cc.y)
                dirscore=max(0.0,1.0-angular_difference(b,desired)/self.config.spatial.direction_tolerance_deg)
            vals.append(0.72*dscore+0.28*dirscore)
            details.append({"boundary_distance_m":float(d),"bearing_from_subject_deg":None if b is None else float(b)})
        return float(np.mean(vals)),True,{"anchors":details}

    def rear_front(self,candidate,anchors,rear=True):
        vals=[]; details=[]
        for a in anchors:
            road=self.ctx.nearest_road_to_property(a)
            if road is None:
                continue
            ap=self.anchor_geometry(a).centroid
            rp=nearest_points(ap,road)[1]
            vx,vy=ap.x-rp.x,ap.y-rp.y
            norm=math.hypot(vx,vy)
            if norm<0.5:
                continue
            ux,uy=vx/norm,vy/norm
            cc=candidate.centroid; wx,wy=cc.x-ap.x,cc.y-ap.y
            projection=wx*ux+wy*uy
            orientation=1.0 if ((projection>0)==rear) else 0.0
            d=candidate.distance(ap)
            dscore=exponential_score(d,self.config.spatial.rear_front_distance_scale_m)
            vals.append(0.72*orientation+0.28*dscore)
            details.append({"projection_m":float(projection),"distance_m":float(d)})
        if not vals:return 0.0,False,{"reason":"nearest road unavailable/indeterminate"}
        return float(np.mean(vals)),True,{"anchors":details}

    def opposite(self,candidate,anchors):
        vals=[];details=[]
        for a in anchors:
            road=self.ctx.nearest_road_to_property(a)
            if road is None:continue
            ap=self.anchor_geometry(a).centroid; cp=candidate.centroid
            crosses=LineString([ap,cp]).intersects(road)
            d=candidate.distance(ap)
            vals.append(0.72*float(crosses)+0.28*exponential_score(d,self.config.spatial.opposite_distance_scale_m))
            details.append({"road_between":bool(crosses),"distance_m":float(d)})
        if not vals:return 0.0,False,{"reason":"nearest road unavailable/indeterminate"}
        return float(np.mean(vals)),True,{"anchors":details}

    def evaluate(self,candidate,constraint:SpatialConstraint,h:AnchorHypothesis):
        anchors=[h.selections.get(aid) for aid in constraint.anchor_ids]
        anchors=[a for a in anchors if a is not None]
        rel=constraint.relation
        if rel=="adjacent_to":return self.adjacent(candidate,anchors)
        if rel=="near":return self.near(candidate,anchors)
        if rel=="between":return self.between(candidate,anchors)
        if rel=="bounded_by":return self.bounded_by(candidate,anchors,constraint.direction)
        if rel=="rear_of":return self.rear_front(candidate,anchors,True)
        if rel=="front_of":return self.rear_front(candidate,anchors,False)
        if rel=="opposite":return self.opposite(candidate,anchors)
        if rel in {"north_of","south_of","east_of","west_of"}:
            return self.directional(candidate,anchors,rel.replace("_of",""))
        if rel in {"accessed_from","part_of"}:return self.near(candidate,anchors)
        return 0.0,False,{"reason":f"unsupported relation {rel}"}

    def score_candidate(self,candidate,parsed:ParsedDescription,h:AnchorHypothesis):
        total_weight=0.0; available_weight=0.0; weighted=0.0; required_contradiction=False; evidence=[]
        for con in parsed.constraints:
            w=self.config.spatial.relation_weights.get(con.relation,1.0)*strength_multiplier(con.strength)
            total_weight+=w
            score,available,detail=self.evaluate(candidate,con,h)
            if available:
                available_weight+=w; weighted+=score*w
                if con.strength=="required" and score<self.config.spatial.required_contradiction_threshold:
                    required_contradiction=True
            evidence.append({
                "relation":con.relation,"anchor_ids":con.anchor_ids,"strength":con.strength,
                "available":available,"score":float(score) if available else None,"detail":detail,
                "original_phrase":con.original_phrase,
            })
        spatial=weighted/available_weight if available_weight>0 else 0.0
        if required_contradiction:
            spatial*=self.config.spatial.required_contradiction_multiplier
        coverage=available_weight/total_weight if total_weight>0 else 0.0
        return float(spatial),float(coverage),required_contradiction,evidence

## 11. Candidate generation across multiple anchor hypotheses

In [ ]:
class CandidateGenerator:
    def __init__(self,inspire:InspireRepository,engine:SpatialEngine,context:GeographicContext,config:AppConfig):
        self.inspire=inspire;self.engine=engine;self.context=context;self.config=config

    def generate(self,parsed:ParsedDescription,hypotheses:list[AnchorHypothesis]) -> list[dict]:
        best_by_id={}
        for hi,h0 in enumerate(hypotheses):
            h=self.context.enrich_hypothesis(h0,parsed)
            bounds=hypothesis_search_bbox(h,parsed,self.config)
            if bounds is None:continue
            g=self.inspire.candidates(bounds,self.config.retrieval.max_inspire_candidates_per_hypothesis)
            for _,row in g.iterrows():
                geom=row.geometry
                spatial,coverage,req_bad,evidence=self.engine.score_candidate(geom,parsed,h)
                iid=str(row["inspire_id"])
                record={
                    "inspire_id":iid,"geometry":geom,
                    "centroid_easting":float(geom.centroid.x),"centroid_northing":float(geom.centroid.y),
                    "area_m2":float(geom.area),"spatial_score":spatial,"coverage_score":coverage,
                    "required_contradiction":bool(req_bad),"constraint_evidence":evidence,
                    "anchor_hypothesis_index":hi,"anchor_hypothesis_score":float(h.normalized_score),
                    "anchor_selections":{
                        aid:(None if c is None else {
                            "source":c.source,"source_id":c.source_id,"uprn":c.uprn,"label":c.label,
                            "easting":c.easting,"northing":c.northing,"lexical_score":c.lexical_score
                        }) for aid,c in h.selections.items()
                    },
                }
                # preliminary score used only to choose evidence for semantic reranking
                record["preliminary_score"]=(
                    self.config.decision.spatial_weight*spatial+
                    self.config.decision.anchor_hypothesis_weight*h.normalized_score+
                    self.config.decision.evidence_coverage_weight*coverage
                )
                if iid not in best_by_id or record["preliminary_score"]>best_by_id[iid]["preliminary_score"]:
                    best_by_id[iid]=record
        return sorted(best_by_id.values(),key=lambda r:r["preliminary_score"],reverse=True)

## 12. LLM semantic evidence adjudication

The LLM sees only **real candidate evidence** produced by OS/HMLR/GIS. It is explicitly forbidden from adding geography from memory.

In [ ]:
SEMANTIC_PROMPT="""
You adjudicate already retrieved HMLR INSPIRE candidate polygons against an England/Wales land description.
All geographic facts in the payload come from OS/HMLR/GIS and are authoritative for this task.
Do NOT introduce external coordinates, addresses, UPRNs, roads or title facts from memory.

For each candidate, assess whether the structured GIS evidence is semantically consistent with the original wording.
Missing/unavailable evidence is UNKNOWN, not a contradiction. Explicit GIS conflicts with a required phrase are important.
Return a 0..1 consistency score, concise supporting points, and actual contradictions only.
"""

class SemanticReasoner:
    def __init__(self,llm:LLMServices,client:OpenAI,config:AppConfig):
        self.llm=llm;self.client=client;self.config=config

    def assess(self,description:str,parsed:ParsedDescription,candidates:list[dict]) -> Optional[SemanticAssessment]:
        if not self.config.llm.semantic_rerank or not candidates:
            return None
        top=candidates[:self.config.llm.max_semantic_candidates]
        payload={
            "original_description":description,
            "parsed":parsed.model_dump(),
            "candidates":[{
                "inspire_id":c["inspire_id"],
                "spatial_score":c["spatial_score"],
                "coverage_score":c["coverage_score"],
                "required_contradiction":c["required_contradiction"],
                "anchor_selections":c["anchor_selections"],
                "constraint_evidence":c["constraint_evidence"],
            } for c in top]
        }
        r=self.client.responses.parse(
            model=self.config.llm.deployment,
            instructions=SEMANTIC_PROMPT,
            input=json_safe(payload),
            text_format=SemanticAssessment,
        )
        return r.output_parsed

## 13. Final scoring, confidence and output

`confidence_score` is a **ranking confidence score**, not a legal probability. Calibrate it empirically using labelled cases.

In [ ]:
def serialise_candidate(c:dict) -> dict:
    geom=c["geometry"]
    lon,lat=BNG_TO_WGS84.transform(c["centroid_easting"],c["centroid_northing"])
    return {
        k:v for k,v in c.items() if k!="geometry"
    } | {
        "centroid_longitude":float(lon),"centroid_latitude":float(lat),
        "geometry_geojson_bng":mapping(geom),
    }


def finalise_candidates(candidates:list[dict],semantic:Optional[SemanticAssessment],config:AppConfig) -> list[dict]:
    sem={x.inspire_id:x for x in semantic.candidates} if semantic else {}
    for c in candidates:
        if c["inspire_id"] in sem:
            s=sem[c["inspire_id"]]
            semantic_score=s.consistency_score
            c["semantic_supporting_points"]=s.supporting_points
            c["semantic_contradictions"]=s.contradictions
        else:
            # Neutral fallback when semantic reranker was not called for this lower-ranked candidate.
            semantic_score=c["spatial_score"]
            c["semantic_supporting_points"]=[];c["semantic_contradictions"]=[]
        c["semantic_score"]=float(semantic_score)
        c["final_score"]=float(
            config.decision.spatial_weight*c["spatial_score"]+
            config.decision.anchor_hypothesis_weight*c["anchor_hypothesis_score"]+
            config.decision.semantic_weight*semantic_score+
            config.decision.evidence_coverage_weight*c["coverage_score"]
        )
    return sorted(candidates,key=lambda r:r["final_score"],reverse=True)


def decide_status(ranked:list[dict],config:AppConfig) -> tuple[str,float]:
    if not ranked:return "UNRESOLVED",0.0
    best=ranked[0]["final_score"]
    second=ranked[1]["final_score"] if len(ranked)>1 else 0.0
    margin=max(0.0,best-second)
    coverage=ranked[0]["coverage_score"]
    margin_component=min(1.0,margin/max(config.decision.minimum_winner_margin,1e-6))
    confidence=(
        config.decision.confidence_score_weight*best+
        config.decision.confidence_margin_weight*margin_component+
        config.decision.confidence_coverage_weight*coverage
    )
    confidence=float(max(0.0,min(1.0,confidence)))
    if best>=config.decision.matched_threshold and margin>=config.decision.minimum_winner_margin:
        return "MATCHED",confidence
    if best>=config.decision.minimum_candidate_threshold:
        return "MULTIPLE_CANDIDATES",confidence
    return "UNRESOLVED",confidence

## 14. End-to-end automated pipeline

In [ ]:
class LandGeolocationPipeline:
    def __init__(self,config:AppConfig=CONFIG,client:OpenAI=azure_client):
        self.config=config
        self.llm=LLMServices(client,config)
        self.addresses=AddressRepository(config)
        self.names=NameRepository(config)
        self.inspire=InspireRepository(config)
        self.context=GeographicContext(config,self.inspire)
        self.anchor_resolver=AnchorResolver(self.addresses,self.names,self.llm,config)
        self.engine=SpatialEngine(self.context,config)
        self.generator=CandidateGenerator(self.inspire,self.engine,self.context,config)
        self.semantic=SemanticReasoner(self.llm,client,config)

    def process(self,description:str,record_id:Optional[str]=None,location_plan_s3_uri:Optional[str]=None) -> PipelineResult:
        record_id=record_id or uuid.uuid4().hex
        if not description or not description.strip():
            return PipelineResult(record_id=record_id,status="INVALID_INPUT",unresolved_reasons=["empty description"])

        warnings=[
            "HMLR INSPIRE polygons are indicative registered freehold extents and are not definitive legal boundaries."
        ]
        if location_plan_s3_uri:
            warnings.append("Location plan was supplied but V1 is text-first and deliberately does not use it.")

        try:
            parsed=self.llm.parse_description(description)
            candidate_sets=self.anchor_resolver.retrieve_all(parsed)
            missing=[aid for aid,rows in candidate_sets.items() if not rows]
            hypotheses=self.anchor_resolver.build_hypotheses(candidate_sets)

            if not hypotheses or all(all(v is None for v in h.selections.values()) for h in hypotheses):
                return PipelineResult(
                    record_id=record_id,status="UNRESOLVED",parsed_description=parsed.model_dump(),
                    unresolved_reasons=["No geographic anchor could be resolved from OS data."],warnings=warnings
                )

            candidates=self.generator.generate(parsed,hypotheses)
            if not candidates:
                reasons=[]
                if missing:reasons.append(f"Unresolved anchors: {missing}")
                reasons.append("No HMLR INSPIRE candidate polygon found in the hypothesis search areas.")
                return PipelineResult(
                    record_id=record_id,status="UNRESOLVED",parsed_description=parsed.model_dump(),
                    anchor_hypotheses=[self._serialise_hypothesis(h) for h in hypotheses],
                    unresolved_reasons=reasons,warnings=warnings
                )

            semantic=None
            try:
                semantic=self.semantic.assess(description,parsed,candidates)
            except Exception as exc:
                warnings.append(f"Semantic rerank failed; deterministic evidence used instead: {exc}")

            ranked=finalise_candidates(candidates,semantic,self.config)
            status,confidence=decide_status(ranked,self.config)
            serial=[serialise_candidate(c) for c in ranked[:20]]
            reasons=[]
            if status!="MATCHED":
                if missing:reasons.append(f"Some anchors were unresolved: {missing}")
                if len(ranked)>1:
                    reasons.append("Top candidates were not sufficiently separated to force a single match.")
                if ranked[0]["coverage_score"]<0.6:
                    reasons.append("A substantial part of the description could not be tested with available GIS evidence.")

            return PipelineResult(
                record_id=record_id,status=status,confidence_score=confidence,
                best_candidate=serial[0] if serial else None,candidates=serial,
                parsed_description=parsed.model_dump(),
                anchor_hypotheses=[self._serialise_hypothesis(h) for h in hypotheses],
                unresolved_reasons=reasons,warnings=warnings
            )
        except Exception as exc:
            logger.exception("Pipeline failure for %s",record_id)
            return PipelineResult(record_id=record_id,status="ERROR",errors=[str(exc)],warnings=warnings)

    @staticmethod
    def _serialise_hypothesis(h:AnchorHypothesis) -> dict:
        return {
            "score":h.normalized_score,
            "selections":{
                aid:(None if c is None else {
                    "source":c.source,"source_id":c.source_id,"uprn":c.uprn,"label":c.label,
                    "easting":c.easting,"northing":c.northing,"lexical_score":c.lexical_score
                }) for aid,c in h.selections.items()
            }
        }

## 15. Single-record and batch usage

In [ ]:
# Create after setting real CONFIG paths and Azure credentials.
# pipeline = LandGeolocationPipeline(CONFIG)

example = """
Land adjoining 12 and 14 High Street, lying to the rear of 10 High Street
and bounded on the east by Mill Lane, Chester.
"""

# result = pipeline.process(example, record_id="ABC123")
# print(json.dumps(result.model_dump(), indent=2, default=str))

In [ ]:
def write_json(uri:str,obj:dict):
    fs,path=fsspec.core.url_to_fs(uri)
    parent=str(Path(path).parent)
    try:fs.makedirs(parent,exist_ok=True)
    except Exception:pass
    with fs.open(path,"w") as f:
        json.dump(obj,f,indent=2,default=str)


def process_dataframe(df:pd.DataFrame,pipeline:LandGeolocationPipeline,output_prefix:Optional[str]=None) -> pd.DataFrame:
    rows=[]
    for _,r in df.iterrows():
        rid=str(r.get("record_id") or uuid.uuid4().hex)
        desc=str(r.get("description") or "")
        plan=r.get("location_plan_s3_uri")
        if pd.isna(plan):plan=None
        result=pipeline.process(desc,rid,plan)
        if output_prefix:
            write_json(output_prefix.rstrip("/")+f"/{rid}.json",result.model_dump())
        rows.append({
            "record_id":rid,"status":result.status,"confidence_score":result.confidence_score,
            "best_inspire_id":result.best_candidate.get("inspire_id") if result.best_candidate else None,
            "candidate_count":len(result.candidates),"errors":len(result.errors),
        })
    return pd.DataFrame(rows)

# Example:
# jobs = pd.read_csv("s3://bucket/jobs.csv")
# summary = process_dataframe(jobs, pipeline, CONFIG.paths.results)
# summary.to_parquet(CONFIG.paths.results.rstrip("/")+"/_summary.parquet", index=False)

## 16. Synthetic tests for the non-LLM core

These tests run without Azure or S3 and exercise the hardest logic: multi-anchor hypothesis coherence, `between`, direction, adjacency and conservative decision behaviour.

In [ ]:
def run_core_unit_tests():
    # Hypothesis coherence test: two matching anchors near each other should beat a geographically distant combination.
    dummy_cfg=AppConfig()
    class DummyLLM: pass
    resolver=AnchorResolver(None,None,DummyLLM(),dummy_cfg)
    a1=[
        AnchorCandidate("a1","address","x","1","10 High St",1000,1000,0.90),
        AnchorCandidate("a1","address","x","2","10 High St elsewhere",50000,50000,0.91),
    ]
    a2=[
        AnchorCandidate("a2","address","x","3","14 High St",1040,1000,0.88),
        AnchorCandidate("a2","address","x","4","14 High St elsewhere",80000,80000,0.89),
    ]
    hs=resolver.build_hypotheses({"a1":a1,"a2":a2})
    best=hs[0]
    assert best.selections["a1"].source_id=="1" and best.selections["a2"].source_id=="3", "coherence beam failed"

    # Minimal context stub for spatial tests.
    class Ctx:
        def nearest_road_to_property(self,c): return None
    engine=SpatialEngine(Ctx(),dummy_cfg)
    h=AnchorHypothesis({
        "a1":AnchorCandidate("a1","address","x","1","A",0,0,1.0),
        "a2":AnchorCandidate("a2","address","x","2","B",100,0,1.0),
    },1,1)
    candidate=box(40,-10,60,10)
    con=SpatialConstraint(relation="between",anchor_ids=["a1","a2"],strength="strong",original_phrase="between A and B")
    s,available,_=engine.evaluate(candidate,con,h)
    assert available and s>0.70, f"between score unexpectedly low: {s}"

    north_poly=box(-10,40,10,60)
    con2=SpatialConstraint(relation="north_of",anchor_ids=["a1"],strength="medium",original_phrase="north of A")
    s2,available,_=engine.evaluate(north_poly,con2,h)
    assert available and s2>0.90, f"north score failed: {s2}"

    touching=box(0,0,20,20)
    h.selections["a1"].parcel_geojson=mapping(box(-20,0,0,20))
    con3=SpatialConstraint(relation="adjacent_to",anchor_ids=["a1"],strength="strong",original_phrase="adjacent to A")
    s3,available,_=engine.evaluate(touching,con3,h)
    assert available and s3>0.99, f"adjacency score failed: {s3}"

    # Conservative final decision: two close candidates should not be MATCHED.
    fake=[
        {"final_score":0.82,"coverage_score":0.9},
        {"final_score":0.79,"coverage_score":0.9},
    ]
    status,_=decide_status(fake,dummy_cfg)
    assert status=="MULTIPLE_CANDIDATES", status
    return "All core unit tests passed"

run_core_unit_tests()

## 17. Production validation checklist

Before production, build a labelled validation set of real descriptions and expected locations. Measure:

1. Anchor top-1 and top-k resolution accuracy.
2. Final candidate top-1 accuracy and top-3 recall.
3. False-match rate (most important for zero-human automation).
4. `UNRESOLVED` and `MULTIPLE_CANDIDATES` rates.
5. Accuracy by description pattern: adjacent, between, rear, bounded by, opposite, directional, rural named properties.
6. Confidence calibration: cases scored 0.8 should actually be correct at roughly the level your business requires before you interpret the score as probability-like.

Do **not** lower thresholds simply to increase the percentage marked `MATCHED`. In an autonomous system, forcing ambiguous records into one answer usually increases false positives.

## 18. Recommended data additions (optional, not required to run V1)

- **OS Open USRN** is useful for named-street geometry and can be preferred over raw Open Roads when descriptions refer to street names.
- **OS Open Roads** remains a good fallback network geometry source.
- **OS Open Rivers** helps with `bounded by river/brook/canal` descriptions.
- **OS Open Names** helps with named places, roads and landmarks, including Welsh/English naming where supplied.

The pipeline intentionally keeps these behind repository interfaces so datasets can be upgraded without rewriting the LLM or scoring stages.

## 19. Official references checked for this notebook

- Microsoft Learn — Azure OpenAI structured outputs: https://learn.microsoft.com/en-us/azure/foundry/openai/how-to/structured-outputs
- Microsoft Learn — Azure OpenAI Responses API: https://learn.microsoft.com/en-us/azure/foundry/openai/how-to/responses
- OS — AddressBase Core documentation: https://docs.os.uk/os-downloads/products/addresses-and-names-portfolio/addressbase-core
- OS — OS Open Names documentation: https://docs.os.uk/os-downloads/products/addresses-and-names-portfolio/os-open-names
- OS — Open Identifiers policy (including Open USRN): https://www.ordnancesurvey.co.uk/products/open-mastermap-programme/open-id-policy
- HMLR — INSPIRE Index Polygons: https://use-land-property-data.service.gov.uk/datasets/inspire
- HMLR — INSPIRE technical guidance / British National Grid: https://use-land-property-data.service.gov.uk/datasets/inspire/tech-guidance

Checked: 18 August 2026.